# Intro

## Trade Direction 

- core.py class Direction: has no option 'BOTH'
- TradeDirection does: there is an option 'BOTH'

1. Direction (in core.py) is the **physical side of one concrete trade** — a single position can't be long and short at once. \
It's stamped on every Trade and Signal, and several pieces of code branch on it assuming exactly one of two values:
- **P&L in PositionState.exit()** — (price − entry) for a long, (entry − price) for a short. A BOTH trade has no defined P&L formula.
- **PositionState.update_peak()** — high-water (max) for longs, low-water (min) for shorts. BOTH has no defined trailing-stop direction.
- **state.enter(direction, …)** opens one position with one side; chart labels read "Long Entry" / "Short Entry".

Adding BOTH here would be a category error — it's not a third side, it's the *absence* of a single side, and every two-way branch above would need an undefined third case.

2. BOTH lives on **TradeDirection** (in trade_configurator.py) instead, because that's a different kind of thing: a **policy/permission over many trades** ("which sides is this run allowed to take"). \
"Both" is meaningful as a permission set; it's meaningless as the sign of one fill.

### The two are orthogonal

| | Direction (core.py) | TradeDirection (trade_configurator.py) |
|---|---|---|
| what | sign of **one** trade | allowed sides for the **run** |
| values | LONG / SHORT | LONG / SHORT / **BOTH** |
| set by | strategy, per signal (computed) | config, once (ACTIVE_TRADE / --direction) |
| BOTH meaningful? | no — a position has one side | yes — it's a filter over all signals |

So the gate (TradeDirection.BOTH) says "allow longs and shorts," while the strategy still emits each individual trade as a definite Direction.LONG or Direction.SHORT. \
Keeping Direction binary is what lets the P&L and trailing-stop math stay branch-clean.


## How does a startegy choose what direction to trade?

**The strategy's own signal logic decides the sign.**

A strategy chooses direction itself, per bar, in its on_bar() logic — there's no central direction picker. \
It evaluates its indicators and calls either state.enter(Direction.LONG, …) or state.enter(Direction.SHORT, …). \
The chosen side is hardcoded into each strategy's entry branches.

The inverse variant (_inv strategy) is a separate class that flips the same signal: enters the opposite side. \

So **choosing direction** happens at the strategy layer in two ways: 
- which signal logic fires the sign, and 
- which variant (base vs _inv) you run

The **TradeDirection gate** (TradingConfig.direction) doesn't choose — it can only veto. \
The strategy still emits a definite Direction.LONG/SHORT; if the run's gate disallows that side, state.enter() returns None and the attempt is counted as a suppressed entry.

So the flow is: \
strategy.on_bar()  →  picks Direction.LONG/SHORT  →  state.enter(direction, …) \
                                                         ├─ side allowed?  → opens the trade \
                                                         └─ side gated/halted? → None (suppressed)

**How to test how a strategy performs only in LONG**

Set the direction gate to LONG — the gate keeps only the strategy's long entries and suppresses every short.

3 ways:
1. In a notebook (recommended — inline, no global edit):
```python
from engine.backtester import Backtester
from engine.strategy_configurator import StrategyConfig
from engine.strategies import SuperTrendStrategy
from engine.trade_configurator import TradingConfig, TradeDirection
from engine.data_configurator import load_data, ACTIVE

df = load_data()
result = Backtester(
    SuperTrendStrategy(StrategyConfig()),
    symbol=ACTIVE.symbol,
    trading_config=TradingConfig(direction=TradeDirection.LONG),   # ← long-only
).run(df, interval=ACTIVE.interval)
print(result.summary())
```

2. Project-wide: \
Edit the ACTIVE_TRADE block in trade_configurator.py → direction = TradeDirection.LONG, then every notebook (and the CLI) inherits it.

3. CLI: \
python -m engine --strategy supertrend --direction long

Compare all three sides in one loop:
```python
for d in (TradeDirection.BOTH, TradeDirection.LONG, TradeDirection.SHORT):
    r = Backtester(SuperTrendStrategy(StrategyConfig()), symbol=ACTIVE.symbol,
                   trading_config=TradingConfig(direction=d)).run(df, interval=ACTIVE.interval)
    print(f"{d.value:5} | P&L {r.total_pnl_bps:+8.1f} bps | {r.total_trades} trades | {r.suppressed_entries} suppressed")
```

Two things to get right:
1. Use the base strategy, not the _inv variant. \
supertrend + direction=long = the strategy's long setups only — exactly what you want. \
supertrend_inv + direction=long tests something different (it disengages an inversion leg) and will print a warning.
2. Long-only is not just "the long trades from the both-direction run." \
Because the engine holds one position at a time, suppressing a short leaves the book flat, which frees it to take a later long the both-run was too busy (in a short) to take. \
So trade count and timing legitimately differ — that's the true long-only equity path, which is what you're after. \
Check result.suppressed_entries (now shown in summary()) to see how many shorts were dropped.

**Conclusion:**
- The strategy decides the side from its indicators
- The config can only allow or block that side, never change it
- If you want the opposite side of a setup: run the _inv variant — not a config flag
- If you want to test a strategy performance on long entries only:  set the direction gate to LONG (via trade_configurator.py)

## Fees: basis points (bps) vs percent (%)

Bybit quotes fees in **%**, but the engine stores and computes everything in **basis points (bps)**. \
These are the same quantity in different units — converting is just a unit swap, not a calculation:

| Unit | Value |
|------|-------|
| 1 bp | 0.01% |
| 100 bps | 1% |
| Bybit taker 0.04% | 4 bps |
| Bybit taker 0.055% | 5.5 bps |

**Conversion:** bps = percent × 100 \
So Bybit's 0.055% → multiply by 100 → store 5.5 bps.

### Why the engine uses bps

P&L is computed in bps, so costs share the same unit and subtract cleanly. In PositionState.exit():

```python
raw_bps = (price - entry) / entry * 10_000   # fraction → bps
trade.pnl_bps = raw_bps - cost_bps           # same unit, plain subtraction
```

The × 10_000 is the fraction → bps conversion (×100 to get %, ×100 again to get bps). \
Using bps avoids tiny decimals like 0.0004 and keeps fees, slippage, and returns on one consistent scale.

## Configurable parameters

- position_size_bps is a single run-level config value, constant for every trade \
  There's no signal-strength or volatility scaling.
- leverage is a single run-level config value, constant for every trade